In [3]:
%pip install datasets pandas pyarrow matplotlib seaborn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip



# 1. Dataset Acquisition and Exploration

## 1.1 Introduction

The first stage of this project involves acquiring existing
agricultural question-and-answer datasets for the development
of the Edo Language Agricultural Advisory Chatbot.

We will use publicly available datasets from Hugging Face
as initial sources of agricultural information.

The purpose of this stage is to obtain relevant agricultural
data, examine its structure, and determine its suitability
for inclusion in our project dataset.

## 1.2 Objectives

- Identify suitable agricultural datasets.
- Download the selected datasets.
- Examine their structure and available fields.
- Inspect sample questions and answers.
- Identify potential data quality issues.

## 1.3 Expected Outcome

At the end of this stage, we should have locally stored
raw datasets and an understanding of their structure,
content, and potential limitations.


In [4]:
import pandas as pd
import os

# Create directories for storing raw datasets
os.makedirs("../data/raw", exist_ok=True)

# Confirm that the directories are available
print("Dataset storage directories are ready.")

Dataset storage directories are ready.



### Explanation of the Implementation

The `pandas` library is imported to support dataset
loading, manipulation, and exploration.

The `os` module is used to interact with the file system.

The `os.makedirs()` function creates the raw data directory
if it does not already exist. The `exist_ok=True` argument
prevents an error when the directory already exists.

This ensures that downloaded datasets have a designated
location and that the original data can be preserved
separately from subsequent processing stages.



## 1.4 Importing Required Libraries

Several Python libraries are required for dataset acquisition
and exploration.

- Pandas: Used for tabular data manipulation and inspection.
- Pathlib: Used to manage project directories and file paths.
- JSON: Used to store dataset metadata.
- Hugging Face Datasets: Used to retrieve datasets from the
  Hugging Face repository.

These libraries provide the basic tools required for
downloading and organizing the project's raw datasets.


In [5]:

from pathlib import Path
from datetime import datetime, timezone
import json
import pandas as pd

from datasets import load_dataset

print("All required libraries imported successfully.")


c:\Users\XXMUGENX\3D Objects\edofarm-ai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All required libraries imported successfully.



## 1.5 Preparing the Dataset Storage Directories

Before downloading any dataset, the project requires
a structured location for storing acquired data.

The following directories will be used:

- raw: Stores original datasets without modification.
- processed: Stores cleaned and standardized datasets.
- translations: Stores English-to-Edo translations.
- metadata: Stores dataset descriptions, sources and
  licensing information.

Maintaining separate directories helps preserve the original
data and makes the processing workflow reproducible.


In [6]:

# Identify the project root directory.
# This works whether the notebook runs from the project
# directory or from its notebooks subdirectory.

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_DIR = CURRENT_DIR.parent
else:
    PROJECT_DIR = CURRENT_DIR

# Define the dataset directories.
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
TRANSLATIONS_DIR = PROJECT_DIR / "data" / "translations"
METADATA_DIR = PROJECT_DIR / "data" / "metadata"

# Create directories if they do not already exist.
for directory in [
    RAW_DIR,
    PROCESSED_DIR,
    TRANSLATIONS_DIR,
    METADATA_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

print("Project directories prepared successfully.")
print(f"Project root: {PROJECT_DIR}")
print(f"Raw data directory: {RAW_DIR}")
print(f"Processed data directory: {PROCESSED_DIR}")
print(f"Translations directory: {TRANSLATIONS_DIR}")
print(f"Metadata directory: {METADATA_DIR}")

Project directories prepared successfully.
Project root: c:\Users\XXMUGENX\3D Objects\edofarm-ai
Raw data directory: c:\Users\XXMUGENX\3D Objects\edofarm-ai\data\raw
Processed data directory: c:\Users\XXMUGENX\3D Objects\edofarm-ai\data\processed
Translations directory: c:\Users\XXMUGENX\3D Objects\edofarm-ai\data\translations
Metadata directory: c:\Users\XXMUGENX\3D Objects\edofarm-ai\data\metadata



## 1.6 Acquisition of Agricultural Datasets

Two initial datasets have been selected for inspection.

### Dataset A: FarmerChat Nigeria

Published by Digital Green, this dataset contains farmer
questions and AI-generated agricultural responses.

The Nigerian subset is particularly relevant because it
contains agricultural questions associated with Nigeria.

### Dataset B: Agronomy QA Pairs

This dataset contains English agricultural questions
and answers covering crop management, soil fertility,
irrigation, pests and diseases, among other topics.

It will serve as a supplementary source of agricultural
questions and answers.

### Acquisition Method

The Hugging Face `load_dataset()` function will be used
to retrieve both datasets.

The original datasets will be preserved separately so
that subsequent cleaning and modifications do not
alter the acquired source data.


In [7]:

# Dataset A: FarmerChat Nigeria
# Try the current repository ID first, followed by the
# alternate identifier shown in the dataset documentation.

farmerchat_ids = [
    "DigiGreen/farmerchat-queries",
    "DigiGreen/farmerchat-queries-v2"
]

farmerchat_dataset = None
farmerchat_source_id = None
farmerchat_errors = []

for dataset_id in farmerchat_ids:
    try:
        print(f"Trying dataset: {dataset_id}")

        farmerchat_dataset = load_dataset(
            dataset_id,
            "nigeria",
            split="train"
        )

        farmerchat_source_id = dataset_id
        print(f"Successfully loaded: {dataset_id}")
        break

    except Exception as error:
        farmerchat_errors.append(
            f"{dataset_id}: {str(error)}"
        )

if farmerchat_dataset is None:
    raise RuntimeError(
        "FarmerChat could not be loaded.\n"
        + "\n".join(farmerchat_errors)
    )


# Dataset B: Agronomy QA Pairs

agronomy_dataset = load_dataset(
    "flamiinngo/agronomy-qa-agriculture",
    split="train"
)

print("\nBoth datasets have been acquired.")

print("FarmerChat records:", len(farmerchat_dataset))
print("Agronomy QA records:", len(agronomy_dataset))


Trying dataset: DigiGreen/farmerchat-queries


Successfully loaded: DigiGreen/farmerchat-queries

Both datasets have been acquired.
FarmerChat records: 11541
Agronomy QA records: 1914



## 1.7 Preserving the Acquired Datasets

After successful acquisition, both datasets will be saved
locally in their original form.

The Hugging Face `save_to_disk()` function preserves the
dataset structure and its associated features.

This ensures that the original data remains available
throughout subsequent processing stages.

The datasets will not be cleaned, translated or modified
at this stage.


In [8]:

# Define storage locations.
farmerchat_path = RAW_DIR / "farmerchat_nigeria"
agronomy_path = RAW_DIR / "agronomy_qa"

# Save FarmerChat without modifying its records.
if not farmerchat_path.exists():
    farmerchat_dataset.save_to_disk(str(farmerchat_path))
    print("FarmerChat dataset saved.")
else:
    print("FarmerChat folder already exists; skipped saving.")

# Save Agronomy QA without modifying its records.
if not agronomy_path.exists():
    agronomy_dataset.save_to_disk(str(agronomy_path))
    print("Agronomy QA dataset saved.")
else:
    print("Agronomy QA folder already exists; skipped saving.")


FarmerChat folder already exists; skipped saving.
Agronomy QA folder already exists; skipped saving.



## 1.8 Initial Dataset Exploration

Dataset exploration involves examining the structure
and contents of the acquired datasets.

The following properties will be investigated:

- Number of records.
- Available column names.
- Data types and field descriptions.
- Sample questions and corresponding answers.

This stage helps determine whether the acquired datasets
contain the information required for the agricultural
advisory chatbot.

It also identifies differences in their structures,
which will be important when combining them later.


In [9]:

# Display the structural information for FarmerChat.

print("FARMERCHAT NIGERIA")
print("=" * 50)

print("Number of records:", len(farmerchat_dataset))
print("Available columns:", farmerchat_dataset.column_names)

display(pd.DataFrame(farmerchat_dataset[:5]))


# Display the structural information for Agronomy QA.

print("\nAGRONOMY QA PAIRS")
print("=" * 50)

print("Number of records:", len(agronomy_dataset))
print("Available columns:", agronomy_dataset.column_names)

display(pd.DataFrame(agronomy_dataset[:5]))


FARMERCHAT NIGERIA
Number of records: 11541
Available columns: ['crop', 'user_country', 'user_geo_level2', 'user_geo_level3', 'query_year_month', 'query', 'response']


,crop,user_country,user_geo_level2,user_geo_level3,query_year_month,query,response
0,generic,Nigeria,Kano,Kano Municipal,2026-02,Should I water the leaves and the base of the ...,Hello there! It's great to hear from you. When...
1,generic,Nigeria,Edo,Etsako West,2026-02,When is the best time to apply fertilizer to m...,"Hello there! It's great to hear from you, and ..."
2,generic,Nigeria,Rivers,Obio/Akpor,2026-02,How can I protect my crops from the current we...,Hello there! 😊 It sounds like you're concerned...
3,generic,Nigeria,Kano,Kano Municipal,2026-02,How does rainwater benefit plant growth during...,Hello! [XXX]. Let's talk about how rainwater b...
4,generic,Nigeria,Rivers,Obio/Akpor,2026-02,What are the benefits of intercropping during ...,Hello there! 😊 Intercropping during the dry se...



AGRONOMY QA PAIRS
Number of records: 1914
Available columns: ['question', 'answer']


,question,answer
0,why is crop rotation important in farming?,This helps to prevent soil erosion and depleti...
1,what is crop rotation,Crop rotation is the practice of growing a ser...
2,what are the different methods of irrigation?,"surface irrigation, drip irrigation, and sprin..."
3,why is soil health vital?,Soil health is critical to crop growth and pro...
4,what are the causes of soil degradation?,"erosion, compaction, and nutrient depletion"



## 1.9 Dataset Quality Assessment

After examining the initial structure of the datasets,
the next step is to assess their quality.

Dataset quality assessment helps determine whether the
acquired records are suitable for further processing.

The assessment will examine:

1. Missing and empty values.
2. Duplicate records.
3. Repeated agricultural questions.
4. Repeated question-and-answer pairs.
5. Question and answer length distributions.

This process is important because duplicated, incomplete
or irrelevant records can affect the quality of our final
training dataset.

The original datasets will remain unchanged. The results
of this assessment will be used to guide subsequent
cleaning operations.


In [10]:

# Convert both acquired Hugging Face datasets into Pandas DataFrames.

farmerchat_df = farmerchat_dataset.to_pandas()
agronomy_df = agronomy_dataset.to_pandas()

# Store the datasets in a dictionary for easier examination.
datasets = {
    "FarmerChat Nigeria": farmerchat_df,
    "Agronomy QA": agronomy_df
}

print("DataFrames prepared successfully.")


DataFrames prepared successfully.



### 1.9.1 Missing and Empty Value Analysis

Missing values refer to fields that contain no recorded
information. Empty text fields may also contain blank
strings instead of actual null values.

Both conditions need to be examined because they can
affect subsequent text processing and model training.

The analysis will calculate the number of null and
empty values in every column of each dataset.


In [11]:

def inspect_missing_values(df, dataset_name):

    results = []

    for column in df.columns:

        null_count = df[column].isna().sum()

        blank_count = df[column].map(
            lambda value: isinstance(value, str) and not value.strip()
        ).sum()

        results.append({
            "column": column,
            "null_values": null_count,
            "empty_strings": blank_count
        })

    report = pd.DataFrame(results)

    print(f"\n{dataset_name}")
    display(report)

    return report


farmerchat_missing = inspect_missing_values(
    farmerchat_df,
    "FarmerChat Nigeria"
)

agronomy_missing = inspect_missing_values(
    agronomy_df,
    "Agronomy QA"
)



FarmerChat Nigeria


,column,null_values,empty_strings
0,crop,0,0
1,user_country,0,0
2,user_geo_level2,0,0
3,user_geo_level3,0,70
4,query_year_month,0,0
5,query,0,0
6,response,0,0



Agronomy QA


,column,null_values,empty_strings
0,question,0,0
1,answer,0,0



### 1.9.2 Duplicate Record Analysis

Duplicate records are entries that contain the same
question-and-answer information or repeated questions.

Duplicates can unnecessarily increase the representation
of certain questions in the training dataset.

The analysis will distinguish between:

- Exact duplicate rows.
- Repeated questions.
- Repeated question-and-answer pairs.

Repeated answers alone will not automatically be
considered duplicates because different questions may
legitimately have the same answer.


In [12]:

def inspect_duplicates(df, question_column, answer_column, dataset_name):

    # Normalize text for comparison without changing the original dataframe.
    normalized_questions = (
        df[question_column]
        .fillna("")
        .astype(str)
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    normalized_answers = (
        df[answer_column]
        .fillna("")
        .astype(str)
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    exact_duplicate_rows = df.duplicated().sum()

    repeated_questions = normalized_questions.duplicated().sum()

    repeated_qa_pairs = pd.DataFrame({
        "question": normalized_questions,
        "answer": normalized_answers
    }).duplicated().sum()

    report = {
        "dataset": dataset_name,
        "total_records": len(df),
        "exact_duplicate_rows": int(exact_duplicate_rows),
        "repeated_questions": int(repeated_questions),
        "repeated_question_answer_pairs": int(repeated_qa_pairs)
    }

    return report


farmerchat_duplicates = inspect_duplicates(
    farmerchat_df,
    "query",
    "response",
    "FarmerChat Nigeria"
)

agronomy_duplicates = inspect_duplicates(
    agronomy_df,
    "question",
    "answer",
    "Agronomy QA"
)

duplicate_report = pd.DataFrame([
    farmerchat_duplicates,
    agronomy_duplicates
])

display(duplicate_report)


,dataset,total_records,exact_duplicate_rows,repeated_questions,repeated_question_answer_pairs
0,FarmerChat Nigeria,11541,0,2,0
1,Agronomy QA,1914,0,0,0



### 1.9.3 Text Length Analysis

Question and answer lengths provide insight into the
structure and complexity of the acquired records.

Very short questions may lack sufficient context, while
extremely long responses may contain information that
requires additional inspection.

This analysis will calculate the minimum, maximum,
average and median lengths of questions and answers.

The results will help establish appropriate text
processing and filtering criteria.


In [13]:

def text_length_analysis(df, question_column, answer_column, dataset_name):

    question_lengths = df[question_column].fillna("").astype(str).str.len()
    answer_lengths = df[answer_column].fillna("").astype(str).str.len()

    report = pd.DataFrame({
        "Question length": question_lengths.describe(),
        "Answer length": answer_lengths.describe()
    })

    print(f"\n{dataset_name} text length statistics")
    display(report)

    return report


farmerchat_lengths = text_length_analysis(
    farmerchat_df,
    "query",
    "response",
    "FarmerChat Nigeria"
)

agronomy_lengths = text_length_analysis(
    agronomy_df,
    "question",
    "answer",
    "Agronomy QA"
)



FarmerChat Nigeria text length statistics


,Question length,Answer length
count,11541.000000,11541.000000
mean,89.430119,2188.781042
std,20.218351,323.302896
min,22.000000,370.000000
25%,76.000000,1990.000000
50%,88.000000,2194.000000
75%,101.000000,2393.000000
max,233.000000,6291.000000



Agronomy QA text length statistics


,Question length,Answer length
count,1914.000000,1914.000000
mean,60.449843,225.745559
std,24.123893,183.538067
min,11.000000,25.000000
25%,44.000000,102.000000
50%,58.000000,180.000000
75%,73.000000,302.000000
max,227.000000,1992.000000


In [14]:

# Combine the missing-value reports.
missing_report = pd.concat(
    [
        farmerchat_missing.assign(dataset="FarmerChat Nigeria"),
        agronomy_missing.assign(dataset="Agronomy QA")
    ],
    ignore_index=True
)

# Save inspection reports.
missing_report.to_csv(
    METADATA_DIR / "missing_values_report.csv",
    index=False
)

duplicate_report.to_csv(
    METADATA_DIR / "duplicate_analysis_report.csv",
    index=False
)

print("Inspection reports saved successfully.")
print("Location:", METADATA_DIR)


Inspection reports saved successfully.
Location: c:\Users\XXMUGENX\3D Objects\edofarm-ai\data\metadata


### 1.9.4 Investigation of Repeated Questions

The initial duplicate analysis identified two repeated
question occurrences in the FarmerChat dataset.

These records will be examined to determine whether they
represent identical questions with different responses
or legitimate repeated questions from different users.

The records will not be automatically deleted.

This investigation helps prevent the removal of potentially
useful information during dataset preparation.

In [15]:
# Normalize questions for comparison.
# This does not modify the original dataset.

farmerchat_questions = (
    farmerchat_df["query"]
    .fillna("")
    .astype(str)
    .str.casefold()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Find every occurrence of a repeated question.
repeated_mask = (
    farmerchat_questions.duplicated(keep=False)
    & farmerchat_questions.ne("")
)

repeated_records = farmerchat_df.loc[
    repeated_mask,
    [
        "crop",
        "user_country",
        "user_geo_level2",
        "user_geo_level3",
        "query_year_month",
        "query",
        "response"
    ]
].copy()

print("Repeated FarmerChat question records:")
display(repeated_records)

Repeated FarmerChat question records:


,crop,user_country,user_geo_level2,user_geo_level3,query_year_month,query,response
6940,generic,Nigeria,Ekiti,Ado-Ekiti,2025-10,What is the purpose of golvet injection for pi...,Hello there! It's great to hear from you again...
6941,generic,Nigeria,Ekiti,Ado-Ekiti,2025-10,What is the purpose of Golvet injection for pi...,Hello! It's great to continue our conversation...
9651,generic,Nigeria,Akwa Ibom,Etim Ekpo,2026-01,What issues might I be facing with my snake go...,Hello there! It sounds like you're having some...
9652,generic,Nigeria,Akwa Ibom,Etim Ekpo,2026-01,What issues might I be facing with my Snake Go...,Hello there! It's great to hear from you again...


### 1.9.5 Geographic Representation Analysis

FarmerChat contains geographic information associated
with the submitted agricultural questions.

The geographic fields will be examined to identify
the distribution of records across the available locations.

Particular attention will be given to records where
the geographic level 2 field contains the value "Edo".

This analysis will help determine how much geographically
relevant Nigerian agricultural information is available
in the dataset.

The geographic fields will be interpreted according to
their original dataset documentation rather than assuming
that every location value represents a particular
administrative level.

In [16]:
# Examine the most frequent locations at each geographic level.

for column in ["user_geo_level2", "user_geo_level3"]:

    print(f"\nDistribution of {column}")

    location_counts = (
        farmerchat_df[column]
        .fillna("Missing")
        .replace(r"^\s*$", "Missing", regex=True)
        .value_counts()
        .head(20)
        .rename_axis("location")
        .reset_index(name="record_count")
    )

    display(location_counts)


# Identify records where geographic level 2 is exactly Edo.
edo_mask = (
    farmerchat_df["user_geo_level2"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.casefold()
    .eq("edo")
)

edo_records = farmerchat_df.loc[edo_mask].copy()

print("\nRecords with geographic level 2 = Edo:", len(edo_records))

print(
    "Percentage of FarmerChat Nigeria:",
    round((len(edo_records) / len(farmerchat_df)) * 100, 2),
    "%"
)

display(
    edo_records[
        [
            "crop",
            "user_geo_level2",
            "user_geo_level3",
            "query",
            "response"
        ]
    ].head(10)
)


Distribution of user_geo_level2


,location,record_count
0,Kano,1701
1,Ogun State,1167
2,Lagos,1145
3,Federal Capital Territory,912
4,Oyo,827
5,Plateau,596
6,Kaduna,498
7,Rivers,494
8,Delta,482
9,Ondo,412



Distribution of user_geo_level3


,location,record_count
0,Abuja Municipal Area Council,758
1,Obio/Akpor,441
2,Eti Osa,408
3,Gezawa,343
4,Jos North,311
5,Kura,260
6,Bebeji,212
7,Nassarawa,208
8,Obafemi Owode,195
9,Egbado North,189



Records with geographic level 2 = Edo: 328
Percentage of FarmerChat Nigeria: 2.84 %


,crop,user_geo_level2,user_geo_level3,query,response
1,generic,Edo,Etsako West,When is the best time to apply fertilizer to m...,"Hello there! It's great to hear from you, and ..."
9,generic,Edo,Oredo,How can I train a young piglet for better care...,Hello there! It's great to hear that you're in...
21,generic,Edo,Etsako West,Which insecticide is effective for controlling...,Hello there! It's great to hear from a farmer ...
41,generic,Edo,Oredo,When is the best time to apply fertilizer to m...,Hello there! It's great to hear that you're fo...
42,generic,Edo,Oredo,What are the duties of an extension agent in a...,Hello there! It's great to hear from you. As y...
57,generic,Edo,Oredo,How can I manage maize pests effectively at th...,"Hello there! It's great to hear from you, and ..."
89,generic,Edo,Egor,What insecticide is recommended for yam during...,Hello there! It's great to hear from a farmer ...
97,generic,Edo,Oredo,How can I manage heat stress in my livestock i...,Hello there! Managing heat stress in your live...
112,generic,Edo,Oredo,When is the best time to apply insecticide on ...,"Hello there! It's great to hear from you, and ..."
114,generic,Edo,Oredo,What strategies can I use to prevent lodging i...,"Hello there! It's great to hear from you, and ..."


### 1.9.5.1 Geographic Subdivision Analysis of Edo State

This analysis examines the geographic subdivisions
represented among the records associated with Edo State.

The purpose is to identify the distribution of
agricultural questions across the available localities.

The results will provide additional insight into the
geographic representation of Edo-associated records.

In [31]:
# Examine geographic subdivisions within Edo State.

edo_subdivision_report = (
    edo_records["user_geo_level3"]
    .fillna("Missing")
    .astype(str)
    .str.strip()
    .replace("", "Missing")
    .value_counts()
    .rename_axis("geographic_subdivision")
    .reset_index(name="record_count")
)

display(edo_subdivision_report)

# Save the results.
edo_subdivision_report.to_csv(
    METADATA_DIR / "edo_geographic_subdivisions.csv",
    index=False
)

print("Edo geographic subdivision report saved.")

,geographic_subdivision,record_count
0,Oredo,118
1,Ikpoba-Okha,94
2,Uhunmwonde,49
3,Ovia North-East,31
4,Esan West,15
5,Egor,6
6,Akoko-Edo,6
7,Etsako East,4
8,Etsako West,2
9,Orhionmwon,2


Edo geographic subdivision report saved.


### Geographic Analysis Findings

The FarmerChat Nigeria dataset contains 11,541 records.

Of these, 328 records (2.84%) are associated with
Edo State according to the user_geo_level2 field.

Edo State is represented alongside other Nigerian
locations, including Kano, Ogun State, Lagos, Rivers
and Delta.

The Edo-associated records include questions on
fertilizer application, pest management, crop
cultivation and livestock management.

Although these records provide useful geographic
context, they should not be interpreted as questions
collected specifically in the Edo language.

The dataset's crop field also contains only the
generic classification, meaning that specific
crop identification requires examination of the
actual question and response text.

### 1.9.6 Agricultural Crop Representation

The crop field provides information about the agricultural
commodities represented in the FarmerChat dataset.

Examining the distribution of crop values will help identify
which crops are frequently discussed and which may be
underrepresented.

Records classified as "generic" will be examined separately
because they do not specify a particular crop.

This analysis will guide the selection of records and help
identify possible gaps in agricultural crop representation.

In [17]:
crop_distribution = (
    farmerchat_df["crop"]
    .fillna("Missing")
    .value_counts()
    .rename_axis("crop")
    .reset_index(name="record_count")
)

print("Top 25 crop categories:")
display(crop_distribution.head(25))

generic_count = (
    farmerchat_df["crop"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.casefold()
    .eq("generic")
    .sum()
)

print("Generic crop records:", generic_count)
print(
    "Generic crop percentage:",
    round((generic_count / len(farmerchat_df)) * 100, 2),
    "%"
)

Top 25 crop categories:


,crop,record_count
0,generic,11541


Generic crop records: 11541
Generic crop percentage: 100.0 %



### 1.9.7 Agricultural Topic Coverage

This analysis examines the agricultural topics represented
in the acquired datasets.

The project identifies seven major agricultural categories:

1. Crop cultivation
2. Pest management
3. Disease management
4. Soil management
5. Irrigation
6. Fertilizer application
7. Harvesting

A preliminary keyword-based approach will be used to
identify questions associated with these categories.

The analysis is exploratory and does not represent
the final intent classification labels.

A question may match multiple categories, while some
questions may not match any of the selected keywords.

The results will help identify gaps in agricultural
topic representation and guide subsequent dataset selection.


In [19]:

import re

# Define preliminary keywords for each agricultural category.

topic_patterns = {
    "crop_cultivation": (
        r"\b(planting|plant|sowing|sow|seed|transplant|"
        r"cultivation|cultivate|spacing|pruning|growing)\b"
    ),

    "pest_management": (
        r"\b(pest|insect|aphid|borer|worm|locust|weevil|"
        r"mite|caterpillar|pesticide)\b"
    ),

    "disease_management": (
        r"\b(disease|fungus|fungal|blight|wilt|rot|"
        r"mildew|rust|virus|bacterial)\b"
    ),

    "soil_management": (
        r"\b(soil|erosion|compaction|ph|nutrient|"
        r"compost|organic matter)\b"
    ),

    "irrigation": (
        r"\b(irrigation|irrigate|watering|water|drip|"
        r"sprinkler|rainfall|drought)\b"
    ),

    "fertilizer_application": (
        r"\b(fertilizer|fertiliser|fertilizing|fertilising|"
        r"urea|npk|manure)\b"
    ),

    "harvesting": (
        r"\b(harvest|harvesting|postharvest|post-harvest)\b"
    )
}


def analyze_topic_coverage(df, question_column, dataset_name):

    questions = (
        df[question_column]
        .fillna("")
        .astype(str)
        .str.casefold()
    )

    results = []

    matched_any = pd.Series(False, index=df.index)

    for category, pattern in topic_patterns.items():

        matches = questions.str.contains(
            pattern,
            regex=True,
            na=False
        )

        matched_any |= matches

        results.append({
            "dataset": dataset_name,
            "category": category,
            "matching_questions": int(matches.sum())
        })

    results.append({
        "dataset": dataset_name,
        "category": "unmatched_by_keywords",
        "matching_questions": int((~matched_any).sum())
    })

    return pd.DataFrame(results)


# Analyze both datasets.

farmerchat_topics = analyze_topic_coverage(
    farmerchat_df,
    "query",
    "FarmerChat Nigeria"
)

agronomy_topics = analyze_topic_coverage(
    agronomy_df,
    "question",
    "Agronomy QA"
)

# Combine and compare the results.

topic_report = pd.concat(
    [farmerchat_topics, agronomy_topics],
    ignore_index=True
)

display(
    topic_report.pivot(
        index="category",
        columns="dataset",
        values="matching_questions"
    )
)


C:\Users\XXMUGENX\AppData\Local\Temp\ipykernel_7420\2434555094.py:57: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  matches = questions.str.contains(
C:\Users\XXMUGENX\AppData\Local\Temp\ipykernel_7420\2434555094.py:57: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  matches = questions.str.contains(


dataset,Agronomy QA,FarmerChat Nigeria
category,,
crop_cultivation,209,1559
disease_management,365,508
fertilizer_application,53,1724
harvesting,71,272
irrigation,98,1277
pest_management,42,818
soil_management,136,376
unmatched_by_keywords,1023,5432



### 1.9.8 Agricultural Response Quality Inspection

This stage examines the quality and structure of the
answers contained in the acquired agricultural datasets.

Particular attention will be given to unusually long
responses, as these may contain excessive conversational
introductions, repeated explanations or irrelevant content.

The longest responses from both datasets will be sampled
and inspected.

The inspection will consider:

1. Relevance to the agricultural question.
2. Clarity and organization of the answer.
3. Presence of useful agricultural recommendations.
4. Excessive repetition or unnecessary information.
5. Potentially unsupported agricultural advice.

The purpose is to determine whether additional
quality-control procedures will be necessary before
the records are included in the final dataset.


In [20]:

# Inspect the five longest answers from both datasets.

def inspect_long_responses(df, question_column, answer_column, dataset_name):

    inspection_df = df.copy()

    # Calculate response lengths.
    inspection_df["answer_length"] = (
        inspection_df[answer_column]
        .fillna("")
        .astype(str)
        .str.len()
    )

    # Select the five longest responses.
    longest = inspection_df.nlargest(5, "answer_length").copy()

    # Limit previews to 1200 characters.
    longest["answer_preview"] = (
        longest[answer_column]
        .fillna("")
        .astype(str)
        .str.slice(0, 1200)
    )

    print(f"\n{dataset_name}: Five longest responses")

    display(
        longest[
            [
                question_column,
                "answer_length",
                "answer_preview"
            ]
        ]
    )


# FarmerChat inspection.
inspect_long_responses(
    farmerchat_df,
    "query",
    "response",
    "FarmerChat Nigeria"
)

# Agronomy QA inspection.
inspect_long_responses(
    agronomy_df,
    "question",
    "answer",
    "Agronomy QA"
)



FarmerChat Nigeria: Five longest responses


,query,answer_length,answer_preview
1109,Can you help me craft 40 more appreciation mes...,6291,Hello there! 🌾 It's wonderful to hear about yo...
2189,Can you provide a list of 50 common weeds alon...,6014,Hello [XXX]! It's wonderful to see your intere...
2190,Can you provide a list of 50 weeds along with ...,5784,Hello! [XXX]. I understand you're looking for ...
7545,What is a project proposal in the context of a...,3414,Hello [XXX]! It's wonderful to see your intere...
2575,What are the best practices for planting rice ...,3293,Hello! It's wonderful to hear that you're inte...



Agronomy QA: Five longest responses


,question,answer_length,answer_preview
859,how can farmers benefit from farmers organization,1992,Knowledge and information sharing: Farming org...
843,How can a farmer control bean weevil in the st...,1690,"Cleaning and hygiene: Before storing beans, en..."
870,name the steps the farmer can use to control p...,1633,Identify the disease: Try to identify the spec...
846,What are the benefits of processing and packag...,1595,Value addition: Processing and packaging add v...
690,What characteristics should a good crop store ...,1493,Good ventilation: Proper air circulation is cr...



### 1.9.9 Investigation of Unmatched Agricultural Questions

Some questions in the datasets were not matched by
the preliminary agricultural keyword patterns.

These records will be sampled for manual inspection.

The purpose is to determine whether the unmatched
questions represent additional agricultural topics,
alternative terminology or questions that require
more detailed classification.

The unmatched records will not automatically be
considered irrelevant or removed.


In [21]:

# Identify questions that do not match any preliminary category.

def get_unmatched_questions(df, question_column):

    questions = (
        df[question_column]
        .fillna("")
        .astype(str)
        .str.casefold()
    )

    matched = pd.Series(False, index=df.index)

    for pattern in topic_patterns.values():

        matched |= questions.str.contains(
            pattern,
            regex=True,
            na=False
        )

    return df.loc[~matched].copy()


farmerchat_unmatched = get_unmatched_questions(
    farmerchat_df,
    "query"
)

agronomy_unmatched = get_unmatched_questions(
    agronomy_df,
    "question"
)

# Examine up to 15 examples from each dataset.
print("FarmerChat unmatched questions:")

display(
    farmerchat_unmatched[["query"]]
    .sample(n=min(15, len(farmerchat_unmatched)), random_state=42)
)

print("\nAgronomy QA unmatched questions:")

display(
    agronomy_unmatched[["question"]]
    .sample(n=min(15, len(agronomy_unmatched)), random_state=42)
)


C:\Users\XXMUGENX\AppData\Local\Temp\ipykernel_7420\3484850698.py:16: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  matched |= questions.str.contains(


FarmerChat unmatched questions:


C:\Users\XXMUGENX\AppData\Local\Temp\ipykernel_7420\3484850698.py:16: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  matched |= questions.str.contains(


,query
11132,How does the current weather in Gezawa affect ...
10401,How long does it take to keep goats and what a...
4677,How can I effectively manage pests on my cucum...
1025,What strategies can I use to prevent lodging i...
9409,Is Magic Force and Lara Force insecticide safe...
8775,When should I vaccinate my noilers and turkeys...
2825,What strategies can I use to prevent lodging i...
10482,What are the best plants to intercrop with mil...
10609,What percentage of egg production drop indicat...
497,How can I protect my cassava from heavy rains ...



Agronomy QA unmatched questions:


,question
886,How does the Agriculture Infrastructure Fund (...
541,why does Propagating cassava through stems ver...
695,can waterlogging cause yellowing in beans plants?
52,what should grapes be harvested?
994,How does Canker survive and spread within citr...
957,Give more information on Stellapps.
701,What are the problems facing farming organizat...
1249,What fertilization practices are recommended f...
116,Why do horses sleep standing up?
1202,What causes Freckle or Black Spot in bananas?



### 1.9.10 Refinement of Agricultural Topic Keywords

The initial keyword-based analysis revealed that several
relevant agricultural questions were not identified.

This occurred because the original keyword vocabulary
did not include common variations, synonyms and alternative
expressions used in agricultural questions.

The keyword patterns will therefore be expanded to
include additional agricultural terminology.

The refined patterns will improve preliminary topic
identification across both datasets.

This remains an exploratory classification method.
Final agricultural intent labels will require further
inspection and annotation.


In [22]:

# Expanded vocabulary for the seven agricultural categories.

topic_patterns = {

    "crop_cultivation": (
        r"\b(?:plant(?:ing|ed)?|sow(?:ing|n)?|seed(?:ling)?s?|"
        r"transplant(?:ing|ed)?|cultivat(?:e|ion|ing)|"
        r"crop rotation|spacing|prun(?:e|ing)|grow(?:ing|th)|"
        r"propagat(?:e|ion|ing)|intercropp(?:ing)?|"
        r"variet(?:y|ies)|lodging|yield)\b"
    ),

    "pest_management": (
        r"\b(?:pests?|insects?|aphids?|borers?|worms?|"
        r"locusts?|weevils?|mites?|caterpillars?|"
        r"insecticides?|pesticides?|herbicides?|weeds?|"
        r"infestation|infestations|pest control)\b"
    ),

    "disease_management": (
        r"\b(?:diseases?|fungus|fungal|blight|wilt|rot|"
        r"mildew|rust|viruses?|bacterial|canker|"
        r"black spot|yellowing|symptoms?|scab)\b"
    ),

    "soil_management": (
        r"\b(?:soil|erosion|compaction|ph|nutrients?|"
        r"compost|organic matter|field capacity|"
        r"waterlogging|soil fertility|drainage|salinity)\b"
    ),

    "irrigation": (
        r"\b(?:irrigat(?:e|ion|ed|ing)|watering|water|"
        r"drip|sprinkler|rainfall|drought|moisture|"
        r"dry spell)\b"
    ),

    "fertilizer_application": (
        r"\b(?:fertili[sz](?:er|ers|ation|ations|e|ed|ing)|"
        r"fertiliser|fertilisers|urea|npk|manure)\b"
    ),

    "harvesting": (
        r"\b(?:harvest(?:s|ed|ing)?|postharvest|"
        r"post-harvest|maturity|ripening)\b"
    )
}

print("Agricultural keyword vocabulary updated.")
print("Number of categories:", len(topic_patterns))


Agricultural keyword vocabulary updated.
Number of categories: 7


In [23]:

# Recalculate topic coverage using the expanded vocabulary.

farmerchat_topics = analyze_topic_coverage(
    farmerchat_df,
    "query",
    "FarmerChat Nigeria"
)

agronomy_topics = analyze_topic_coverage(
    agronomy_df,
    "question",
    "Agronomy QA"
)

# Combine both reports.
refined_topic_report = pd.concat(
    [farmerchat_topics, agronomy_topics],
    ignore_index=True
)

# Display the results.
refined_topic_comparison = refined_topic_report.pivot(
    index="category",
    columns="dataset",
    values="matching_questions"
)

display(refined_topic_comparison)

# Save the refined report.
refined_topic_report.to_csv(
    METADATA_DIR / "refined_topic_coverage.csv",
    index=False
)


dataset,Agronomy QA,FarmerChat Nigeria
category,,
crop_cultivation,350,2816
disease_management,469,731
fertilizer_application,110,1848
harvesting,98,372
irrigation,106,1342
pest_management,135,2107
soil_management,151,513
unmatched_by_keywords,686,3211



### 1.9.11 Agricultural Category Overlap Analysis

The preliminary agricultural topic analysis allows
a question to match more than one agricultural category.

Consequently, the total category matches may exceed
the total number of questions in a dataset.

This section examines the distribution of questions
according to the number of agricultural categories
matched by their keywords.

The analysis will identify:

- Questions matching exactly one category.
- Questions matching multiple categories.
- Questions matching three or more categories.
- Questions that do not match any category.

This is important for understanding category overlap
before assigning final agricultural intent labels.


In [24]:

def build_topic_match_matrix(df, question_column):

    questions = (
        df[question_column]
        .fillna("")
        .astype(str)
        .str.casefold()
    )

    # Create one Boolean column per agricultural category.
    matches = {}

    for category, pattern in topic_patterns.items():

        matches[category] = questions.str.contains(
            pattern,
            regex=True,
            na=False
        )

    match_matrix = pd.DataFrame(matches)

    # Count how many categories each question matches.
    match_matrix["matched_category_count"] = (
        match_matrix.sum(axis=1)
    )

    return match_matrix


# Build category match matrices for both datasets.

farmerchat_match_matrix = build_topic_match_matrix(
    farmerchat_df,
    "query"
)

agronomy_match_matrix = build_topic_match_matrix(
    agronomy_df,
    "question"
)


# Examine the distribution of category matches.

def display_overlap_distribution(match_matrix, dataset_name):

    distribution = (
        match_matrix["matched_category_count"]
        .value_counts()
        .sort_index()
        .rename_axis("number_of_categories_matched")
        .reset_index(name="number_of_questions")
    )

    print(f"\n{dataset_name}")
    display(distribution)


display_overlap_distribution(
    farmerchat_match_matrix,
    "FarmerChat Nigeria"
)

display_overlap_distribution(
    agronomy_match_matrix,
    "Agronomy QA"
)



FarmerChat Nigeria


,number_of_categories_matched,number_of_questions
0,0,3211
1,1,6974
2,2,1313
3,3,43



Agronomy QA


,number_of_categories_matched,number_of_questions
0,0,686
1,1,1051
2,2,163
3,3,14



### 1.9.12 Unique Agricultural Question Coverage

This section calculates the number and percentage of
unique questions that match at least one agricultural
category.

Unlike individual category counts, this measurement
ensures that a question matching multiple categories
is counted only once.

The results provide a more accurate representation
of the overall keyword-based agricultural coverage.


In [27]:

def calculate_unique_coverage(match_matrix, dataset_name, total_records):

    matched_count = (
        match_matrix["matched_category_count"] > 0
    ).sum()

    unmatched_count = total_records - matched_count

    coverage_percentage = (
        matched_count / total_records
    ) * 100

    report = {
        "dataset": dataset_name,
        "total_records": total_records,
        "records_matched_at_least_one_category": int(matched_count),
        "unmatched": int(unmatched_count),
        "coverage_percentage": round(coverage_percentage, 2)
    }

    return report


coverage_results = pd.DataFrame([
    calculate_unique_coverage(
        farmerchat_match_matrix,
        "FarmerChat Nigeria",
        len(farmerchat_df)
    ),

    calculate_unique_coverage(
        agronomy_match_matrix,
        "Agronomy QA",
        len(agronomy_df)
    )
])

display(coverage_results)


,dataset,total_records,records_matched_at_least_one_category,unmatched,coverage_percentage
0,FarmerChat Nigeria,11541,8330,3211,72.18
1,Agronomy QA,1914,1228,686,64.16


In [28]:

# Save the category overlap reports.

farmerchat_match_matrix.to_csv(
    METADATA_DIR / "farmerchat_topic_matches.csv",
    index=False
)

agronomy_match_matrix.to_csv(
    METADATA_DIR / "agronomy_topic_matches.csv",
    index=False
)

coverage_results.to_csv(
    METADATA_DIR / "unique_topic_coverage.csv",
    index=False
)

print("Topic overlap and unique coverage reports saved.")


Topic overlap and unique coverage reports saved.


## 1.10 Dataset Source Documentation

This section documents the sources of the acquired
agricultural datasets.

For reproducibility and responsible data usage, each
source is recorded with its repository, licence,
language, record count and known limitations.

This information will be retained throughout the
dataset construction process.

The source register will also support appropriate
attribution and verification of dataset reuse permissions.

In [29]:
source_register = pd.DataFrame([
    {
        "dataset_name": "FarmerChat Nigeria",
        "platform": "Hugging Face",
        "repository": "DigiGreen/farmerchat-queries",
        "source_url": "https://huggingface.co/datasets/DigiGreen/farmerchat-queries",
        "license": "CC BY 4.0",
        "language": "English",
        "record_count": len(farmerchat_df),
        "fields": ", ".join(farmerchat_df.columns),
        "limitations": (
            "AI-generated responses; questions rephrased by "
            "an intent-classification model; crop field is generic."
        )
    },
    {
        "dataset_name": "Agronomy QA Pairs",
        "platform": "Hugging Face",
        "repository": "flamiinngo/agronomy-qa-agriculture",
        "source_url": "https://huggingface.co/datasets/flamiinngo/agronomy-qa-agriculture",
        "license": "Apache 2.0",
        "language": "English",
        "record_count": len(agronomy_df),
        "fields": ", ".join(agronomy_df.columns),
        "limitations": (
            "Indian regional bias; answers are not individually "
            "verified by an agronomist."
        )
    }
])

display(source_register)

source_register.to_csv(
    METADATA_DIR / "source_register.csv",
    index=False
)

print("Dataset source register saved successfully.")

,dataset_name,platform,repository,source_url,license,language,record_count,fields,limitations
0,FarmerChat Nigeria,Hugging Face,DigiGreen/farmerchat-queries,https://huggingface.co/datasets/DigiGreen/farm...,CC BY 4.0,English,11541,"crop, user_country, user_geo_level2, user_geo_...",AI-generated responses; questions rephrased by...
1,Agronomy QA Pairs,Hugging Face,flamiinngo/agronomy-qa-agriculture,https://huggingface.co/datasets/flamiinngo/agr...,Apache 2.0,English,1914,"question, answer",Indian regional bias; answers are not individu...


Dataset source register saved successfully.


In [30]:
coverage_results = coverage_results.rename(
    columns={
        "uniquely_matched":
        "records_matched_at_least_one_category"
    }
)

coverage_results.to_csv(
    METADATA_DIR / "unique_topic_coverage.csv",
    index=False
)

print("Corrected topic coverage report saved.")

Corrected topic coverage report saved.
